# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published on: {dataset.metadata.datePublished}")
print(f"Authors: {[author['@id'] for author in dataset.metadata.author]}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

We query the Croissant schema using `mlcroissant` to retrieve all record sets and relevant field information, referencing entities strictly by `@id`.

In [ ]:
# List all available record set @id's
record_sets = dataset.record_sets
print("Available Record Sets in this dataset:")
for rset in record_sets:
    print(f"- @id: {rset['@id']}   |   name: {rset.get('name', 'N/A')}")

# For the first record set, list the fields and columns (@id) it contains
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    print(f"\nFields in record set {main_record_set_id}:")
    fields = dataset.fields(record_set=main_record_set_id)
    for field in fields:
        print(f"- Field @id: {field['@id']}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")
        if 'column' in field:
            columns = field['column']
            for col in columns:
                print(f"    - Column @id: {col['@id']}, name: {col.get('name', 'N/A')}, dataType: {col.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. Use `@id` from the record set overview above.

We'll use the main survey record set for further steps.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rset['@id'] for rset in dataset.record_sets]
print("Extracting data for the following record sets:", record_set_ids)

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"DataFrame columns for record set {rs_id}:")
        print(dataframes[rs_id].columns.tolist())
        print(dataframes[rs_id].head(), '\n')
    else:
        print(f"No records found for {rs_id}")
# For subsequent exploration, pick the main record set
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will use the GAD-7 total score field (if present) as our numeric field (referenced by its `@id`). We'll also group by village or similar demographic field.

In [ ]:
# Example: Select numeric and demographic fields by @id
# Replace these IDs with actual ones as discovered above.

# Let's try to infer the relevant field IDs based on column names
numeric_field_id = None
demographic_field_id = None
for col in df.columns:
    if 'gad7' in col.lower() or 'GAD-7' in col:
        numeric_field_id = col
    if 'village' in col.lower():
        demographic_field_id = col

if numeric_field_id is None:
    # Fallback: Use PHQ-9 if GAD-7 not present
    for col in df.columns:
        if 'phq9' in col.lower():
            numeric_field_id = col

print(f"Numeric field selected: {numeric_field_id}")
print(f"Demographic/group field selected: {demographic_field_id}")

# Data cleaning: ensure numeric values
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Remove outliers
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id, demographic_field_id]].head())

# Normalize the numeric field for filtered records
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print("Normalized field example:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group data by demographic field (e.g., village)
if demographic_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(demographic_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {demographic_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of GAD-7 (or selected numeric field) scores
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field_id} Scores")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Compare score by village (or demographic field)
if demographic_field_id:
    plt.figure(figsize=(12, 6))
    sns.boxplot(x=demographic_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {demographic_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded and reviewed the A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library. We examined the structure of the Croissant schema, extracted survey records, and performed simple EDA on mental health scores and their distribution by village. This workflow demonstrates how to use record set and field `@id`s for consistent access and reproducible analyses with FAIR datasets.